# 🚀 RL Trading — Main Notebook

Entry point cho training và inference.

## 1. Training

In [1]:
from src.logging_config import setup_logging
from src.config import Config
from src.training.trainer import Trainer

# Đọc configs/logging.yaml: console WARNING+, file DEBUG chia nhỏ 5MB
log_dir = setup_logging()
print(f"Log dir: {log_dir}")

cfg = Config.load("configs/")
trainer = Trainer(cfg)
trainer.run()

print(f"\n✅ Output directory: {trainer.run_dir}")

Log dir: artifacts\logs\train_20260529_001206
[Data] VNM.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VNM.csv Close range: 47.97 → 88.46
[Split] Train=1028 | Val=129 | Test=129
[Data] FPT.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] FPT.csv Close range: 22.91 → 131.67
[Split] Train=1028 | Val=129 | Test=129
[Data] VIC.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VIC.csv Close range: 19.95 → 169.90
[Split] Train=1028 | Val=129 | Test=129
[Data] HPG.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] HPG.csv Close range: 9.00 → 32.65
[Split] Train=1028 | Val=129 | Test=129


Training:   0%|          | 0/500 [00:00<?, ?ep/s]

KeyboardInterrupt: 

## 2. Evaluate (Test Set)

In [ ]:
# Evaluate trên test set với best model
test_results = trainer.evaluate()

# Export ROI table
trainer.export_roi_table(test_results)

## 3. Biểu đồ

In [ ]:
# Vẽ training curves + per-stock charts
# Tất cả output vào: outputs/output_<timestamp>/charts/
trainer.generate_charts(test_results)

print(f"Charts → {trainer.charts_dir}")

## 4. Inference (placeholder)

In [ ]:
from src.config import Config
from src.inference.inferencer import Inferencer

cfg = Config.load("configs/")
infer = Inferencer(cfg, model_path=str(trainer.weights_dir / "best_model.pkl"))

# Predict cho 1 stock
result = infer.predict("data/VNM.csv")
print(result)

## 5. Hyperparameter Sweep

Vòng lặp train từng bộ hyperparameter trọng tâm.  
Dùng `cfg.override()` để thay đổi giá trị trong memory — không sửa file config.

In [2]:
# ════════════════════════════════════════════════════════════════
# Define experiments — mỗi dict là 1 run với overrides cụ thể
# Chỉ thay đổi 1 param (hoặc 1 cặp liên quan) mỗi lần
# ════════════════════════════════════════════════════════════════

EXPERIMENTS = [
    {"name": "tau_005", "overrides": {"agent.tau": 0.005}},
    {"name": "tau_01",  "overrides": {"agent.tau": 0.01}},

    # ── Tier 2: eps_decay (exploration schedule) ────────────────
    {"name": "eps_decay_990", "overrides": {"agent.eps_decay": 0.990}},
    {"name": "eps_decay_995", "overrides": {"agent.eps_decay": 0.995}},
    {"name": "eps_decay_997", "overrides": {"agent.eps_decay": 0.997}},
    {"name": "eps_decay_999", "overrides": {"agent.eps_decay": 0.999}},
]

print(f"Total experiments: {len(EXPERIMENTS)}")

Total experiments: 6


In [3]:
import json
from src.logging_config import setup_logging
from src.config import Config
from src.training.trainer import Trainer

setup_logging()

# ════════════════════════════════════════════════════════════════
# Sweep loop — train từng experiment
# ════════════════════════════════════════════════════════════════
results_summary = []

for i, exp in enumerate(EXPERIMENTS):
    print(f"\n{'='*60}")
    print(f"  [{i+1}/{len(EXPERIMENTS)}] Experiment: {exp['name']}")
    print(f"  Overrides: {exp['overrides']}")
    print(f"{'='*60}")

    # Load fresh config mỗi lần (reset về baseline)
    cfg = Config.load("configs/")

    # Override hyperparameters
    for key, val in exp["overrides"].items():
        cfg.override(key, val)

    # Train
    try:
        trainer = Trainer(cfg)
        trainer.run()

        # Collect result từ logs.json
        with open(trainer.run_dir / "logs.json", encoding="utf-8") as f:
            run_log = json.load(f)

        results_summary.append({
            "experiment": exp["name"],
            "overrides": exp["overrides"],
            "run_dir": str(trainer.run_dir),
            "test_metrics": run_log["result"].get("test_metrics", {}),
            "best_score": run_log["result"]["best_score"],
            "elapsed_min": run_log["elapsed_minutes"],
        })
        print(f"  ✅ Done: score={run_log['result']['best_score']:.4f}")

    except Exception as e:
        print(f"  ❌ Failed: {e}")
        results_summary.append({
            "experiment": exp["name"],
            "overrides": exp["overrides"],
            "run_dir": "FAILED",
            "test_metrics": {},
            "best_score": 0,
            "elapsed_min": 0,
        })

print(f"\n\n{'='*60}")
print(f"  All {len(EXPERIMENTS)} experiments completed!")
print(f"{'='*60}")


  [1/6] Experiment: tau_005
  Overrides: {'agent.tau': 0.005}
[Data] VNM.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VNM.csv Close range: 47.97 → 88.46
[Split] Train=1028 | Val=129 | Test=129
[Data] FPT.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] FPT.csv Close range: 22.91 → 131.67
[Split] Train=1028 | Val=129 | Test=129
[Data] VIC.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VIC.csv Close range: 19.95 → 169.90
[Split] Train=1028 | Val=129 | Test=129
[Data] HPG.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] HPG.csv Close range: 9.00 → 32.65
[Split] Train=1028 | Val=129 | Test=129


Training:   0%|          | 0/500 [00:00<?, ?ep/s]

[Chart] 3 → artifacts\outputs\output_20260529_001338\charts\training_curves.png
[Chart] 1 → artifacts\outputs\output_20260529_001338\charts\VNM/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260529_001338\charts\VNM/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260529_001338\charts\VNM/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260529_001338\charts\VNM/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260529_001338\charts\VNM/0_dashboard.png
[Charts] 5 biểu đồ → artifacts\outputs\output_20260529_001338\charts\VNM/
[Chart] 1 → artifacts\outputs\output_20260529_001338\charts\FPT/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260529_001338\charts\FPT/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260529_001338\charts\FPT/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260529_001338\charts\FPT/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260529_001338\charts\FPT/0_da

Training:   0%|          | 0/500 [00:00<?, ?ep/s]

[Chart] 3 → artifacts\outputs\output_20260529_013135\charts\training_curves.png
[Chart] 1 → artifacts\outputs\output_20260529_013135\charts\VNM/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260529_013135\charts\VNM/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260529_013135\charts\VNM/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260529_013135\charts\VNM/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260529_013135\charts\VNM/0_dashboard.png
[Charts] 5 biểu đồ → artifacts\outputs\output_20260529_013135\charts\VNM/
[Chart] 1 → artifacts\outputs\output_20260529_013135\charts\FPT/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260529_013135\charts\FPT/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260529_013135\charts\FPT/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260529_013135\charts\FPT/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260529_013135\charts\FPT/0_da

...

[Chart] 3 → artifacts\outputs\output_20260529_031725\charts\training_curves.png
[Chart] 1 → artifacts\outputs\output_20260529_031725\charts\VNM/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260529_031725\charts\VNM/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260529_031725\charts\VNM/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260529_031725\charts\VNM/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260529_031725\charts\VNM/0_dashboard.png
[Charts] 5 biểu đồ → artifacts\outputs\output_20260529_031725\charts\VNM/
[Chart] 1 → artifacts\outputs\output_20260529_031725\charts\FPT/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260529_031725\charts\FPT/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260529_031725\charts\FPT/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260529_031725\charts\FPT/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260529_031725\charts\FPT/0_da

Training:   0%|          | 0/500 [00:00<?, ?ep/s]

[Chart] 3 → artifacts\outputs\output_20260529_035726\charts\training_curves.png
[Chart] 1 → artifacts\outputs\output_20260529_035726\charts\VNM/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260529_035726\charts\VNM/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260529_035726\charts\VNM/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260529_035726\charts\VNM/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260529_035726\charts\VNM/0_dashboard.png
[Charts] 5 biểu đồ → artifacts\outputs\output_20260529_035726\charts\VNM/
[Chart] 1 → artifacts\outputs\output_20260529_035726\charts\FPT/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260529_035726\charts\FPT/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260529_035726\charts\FPT/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260529_035726\charts\FPT/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260529_035726\charts\FPT/0_da

Training:   0%|          | 0/500 [00:00<?, ?ep/s]

[Chart] 3 → artifacts\outputs\output_20260529_045129\charts\training_curves.png
[Chart] 1 → artifacts\outputs\output_20260529_045129\charts\VNM/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260529_045129\charts\VNM/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260529_045129\charts\VNM/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260529_045129\charts\VNM/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260529_045129\charts\VNM/0_dashboard.png
[Charts] 5 biểu đồ → artifacts\outputs\output_20260529_045129\charts\VNM/
[Chart] 1 → artifacts\outputs\output_20260529_045129\charts\FPT/1_price_signals.png
[Chart] 2 → artifacts\outputs\output_20260529_045129\charts\FPT/2_equity_drawdown.png
[Chart] 3 → artifacts\outputs\output_20260529_045129\charts\FPT/3_training_curves.png
[Chart] 4 → artifacts\outputs\output_20260529_045129\charts\FPT/4_trade_analysis.png
[Chart] Dashboard → artifacts\outputs\output_20260529_045129\charts\FPT/0_da

In [4]:
import pandas as pd
import os

# ════════════════════════════════════════════════════════════════
# Summary table — so sánh tất cả experiments
# ════════════════════════════════════════════════════════════════
rows = []
for r in results_summary:
    avg = r["test_metrics"].get("average", {})
    rows.append({
        "Experiment": r["experiment"],
        "Return%": avg.get("return_pct", 0),
        "Sharpe": avg.get("sharpe", 0),
        "WinRate%": avg.get("win_rate", 0),
        "MaxDD%": avg.get("max_dd_pct", 0),
        "N_trades": avg.get("n_trades", 0),
        "PF": avg.get("profit_factor", 0),
        "Score": r["best_score"],
        "Time(min)": r["elapsed_min"],
    })

df_summary = pd.DataFrame(rows)

# Sort by Sharpe (chỉ số quan trọng nhất)
df_summary = df_summary.sort_values("Sharpe", ascending=False).reset_index(drop=True)

# Export CSV
os.makedirs("artifacts/outputs", exist_ok=True)
df_summary.to_csv("artifacts/outputs/hyperparam_sweep_results.csv", index=False)

print("\n📊 Hyperparameter Sweep Results (sorted by Sharpe):")
print("="*80)
display(df_summary)
print(f"\n💾 Saved → artifacts/outputs/hyperparam_sweep_results.csv")


📊 Hyperparameter Sweep Results (sorted by Sharpe):


,Experiment,Return%,Sharpe,WinRate%,MaxDD%,N_trades,PF,Score,Time(min)
0,tau_01,6.03,0.6008,59.7,-5.40,19,2.17,0.3456,77.78
1,eps_decay_999,3.13,0.2262,59.9,-6.39,17,6.90,0.4455,65.08
2,tau_005,-0.46,-0.2098,65.9,-7.90,16,1.98,0.4692,77.93
3,eps_decay_995,-0.77,-0.3452,54.1,-8.04,17,1.62,0.3710,40.01
4,eps_decay_997,-5.57,-1.3633,50.1,-11.44,26,1.05,0.4375,54.03
5,eps_decay_990,-5.99,-4.1126,42.7,-7.26,16,0.75,0.3962,27.99



💾 Saved → artifacts/outputs/hyperparam_sweep_results.csv
